<a href="https://colab.research.google.com/github/RMartinod/Computer-Graphics-Using-Python/blob/main/Chapter23_Exercise_solutions4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**23.15. Solutions to exercises in Chapter 18**

**E.18.1.** In Section 18.5. (Closed Bezier surfaces in 3D) we followed an example that encodes and plots the closed Bézier surface over the v-direction. Now, we propose code the closed Bézier surface over the u-direction.

In [ ]:
import numpy as np, plotly.graph_objects as go, math
i, j = np.linspace(-2*np.pi/3,2*np.pi/3,5), np.linspace(-4*np.pi/5,4*np.pi/5,10)
I, J = np.meshgrid(i,j)
X = np.cos(2*np.pi/3)*np.sinh(I)*np.sin(J) + np.sin(2*np.pi/3)*np.cosh(I)*np.cos(J)
Y =-np.cos(2*np.pi/3)*np.sinh(I)*np.cos(J) + np.sin(2*np.pi/3)*np.cosh(I)*np.sin(J)
Z = J*np.cos(2*np.pi/3) + I*np.sin(2*np.pi/3)
pxu0 = X[0,:].reshape(1,-1)
pyu0 = Y[0,:].reshape(1,-1)
pzu0 = Z[0,:].reshape(1,-1)
X = np.concatenate((X,pxu0))
Y = np.concatenate((Y,pyu0))
Z = np.concatenate((Z,pzu0))
u  = np.transpose(np.linspace(0,1,60).reshape(1, -1))
v  = np.transpose(np.linspace(0,1,60).reshape(1, -1))
m, n = X.shape
Bu = np.zeros((len(u),m))
for i in range(m):
  b = math.comb(m-1,i)*u**i*(1-u)**(m-i-1)
  Bu[:,i] = b[:,0]
Bv = np.zeros((len(v),n))
for j in range(n):
  b = math.comb(n-1,j)*v**j*(1-v)**(n-j-1)
  Bv[:,j] = b[:,0]
px = Bu @ X @ np.transpose(Bv)
py = Bu @ Y @ np.transpose(Bv)
pz = Bu @ Z @ np.transpose(Bv)
fig = go.Figure(go.Surface(x=X,y=Y,z=Z,opacity=.1,colorscale ='ylorrd'))
fig.add_trace(go.Surface(x=px,y=py,z=pz,opacity=1,colorscale='solar'))

**23.16. Solutions to exercises in Chapter 19**

**E.19.1.** In Section 10.4 (see Figure 10-11), we built the basic shape of our aircraft model. We propose to model an aerodynamic shape of our aircraft applying a spline surface

In [ ]:
import numpy as np, plotly.graph_objects as go
u, v = np.linspace(-np.pi/2,np.pi/2,20), np.linspace(0,1,11),
U, V = np.meshgrid(u,v)
Ynose, Znose = 4*V*np.cos(U), 4*V*np.sin(U)
Xnose = 30 -(Ynose**2+Znose**2)/2 + (Ynose[-1]**2+Znose[-1]**2)/2
u = np.linspace(-np.pi/2,np.pi/2,20)
v = np.array([0,-8,-14,-23,-23.5,-23.5,-30,-32,-36,-36,-36])
U,V = np.meshgrid(u,v)
Xbody, Ybody, Zbody = V+30, 4*np.cos(U), 4*np.sin(U)
Ybody[2,6]+=4; Ybody[3,6]+=20; Ybody[4,6]+=21; Xbody[4,6]+=-5
Zbody[3,6]+=3; Zbody[4,6]+=4; Zbody[7,-1]+=1; Zbody[8:11,-1]+=5
Xbody[-1,-1]-=2; Ybody[8:11,-4]+=6; Xbody[-1,-4]-=2
m,n = np.shape(Xnose)
S1nose, S1body  = np.ones((4,m*n)), np.ones((4,m*n))
for row in range(m):
  S1nose[0:3,row*n:(row+1)*n] = [Xnose[row,:],Ynose[row,:],Znose[row,:]]
  S1body[0:3,row*n:(row+1)*n] = [Xbody[row,:],Ybody[row,:],Zbody[row,:]]
RfM = np.array([[1, 0, 0, 0],[ 0,-1, 0, 0],[ 0, 0, 1, 0],[ 0, 0, 0, 1]])
S2nose = RfM@S1nose
S2body = RfM@S1body
X2nose, Y2nose, Z2nose = np.zeros((m,n)), np.zeros((m,n)), np.zeros((m,n))
X2body, Y2body, Z2body = np.zeros((m,n)), np.zeros((m,n)), np.zeros((m,n))
for row in range(m):
  X2nose[row,:],Y2nose[row,:],Z2nose[row,:] = S2nose[0,row*n:(row+1)*n], S2nose[1,row*n:(row+1)*n], S2nose[2,row*n:(row+1)*n]
  X2body[row,:],Y2body[row,:],Z2body[row,:] = S2body[0,row*n:(row+1)*n], S2body[1,row*n:(row+1)*n], S2body[2,row*n:(row+1)*n]
Xplane = np.concatenate((np.concatenate((Xnose,X2nose),axis = 1),np.concatenate((Xbody,X2body),axis = 1)))
Yplane = np.concatenate((np.concatenate((Ynose,Y2nose),axis = 1),np.concatenate((Ybody,Y2body),axis = 1)))
Zplane = np.concatenate((np.concatenate((Znose,Z2nose),axis = 1),np.concatenate((Zbody,Z2body),axis = 1)))
fig = go.Figure(go.Surface(x=Xplane,y=Yplane,z=Zplane,opacity=1,colorscale ='PuBu'))
fig.update_layout(scene = dict(aspectratio = dict(x=2, y=1.8, z=.5)))

In [ ]:
pxv0, pyv0, pzv0 = Xplane[0:2,:], Yplane[0:2,:], Zplane[0:2,:]
X, Y, Z = np.concatenate((Xplane,pxv0)), np.concatenate((Yplane,pyv0)), np.concatenate((Zplane,pzv0))
pxu0, pyu0, pzu0 = Xplane[:,0:2].reshape(-1,2), Yplane[:,0:2].reshape(-1,2), Zplane[:,0:2].reshape(-1,2)
X = np.concatenate((Xplane,pxu0),axis=1)
Y = np.concatenate((Yplane,pyu0),axis=1)
Z = np.concatenate((Zplane,pzu0),axis=1)
u  = np.transpose(np.linspace(0,1,20).reshape(1, -1))
v  = np.transpose(np.linspace(0,1,20).reshape(1, -1))
Un = np.concatenate((u**2,u,np.ones(np.shape(u))),axis=1)
Vm = np.concatenate((u**2,u,np.ones(np.shape(v))),axis=1)
Mn = (1/2)*np.array([[ 1,-2, 1], [-2, 2, 0], [ 1, 1, 0]])
Mm = (1/2)*np.array([[ 1,-2, 1], [-2, 2, 0], [ 1, 1, 0]])
n, m = X.shape
px = np.zeros((len(u)*(n-len(Mn)+1), len(v)*(m-len(Mm)+1)))
py = np.zeros((len(u)*(n-len(Mn)+1), len(v)*(m-len(Mm)+1)))
pz = np.zeros((len(u)*(n-len(Mn)+1), len(v)*(m-len(Mm)+1)))
for i in range (n-len(Mn)+1):
  for j in range (m-len(Mm)+1):
     XStage = X[i:i+len(Mn),j:j+len(Mm)]
     YStage = Y[i:i+len(Mn),j:j+len(Mm)]
     ZStage = Z[i:i+len(Mn),j:j+len(Mm)]
     pxStage = Un @ Mn @ XStage @ np.transpose(Mm) @ np.transpose(Vm)
     pyStage = Un @ Mn @ YStage @ np.transpose(Mm) @ np.transpose(Vm)
     pzStage = Un @ Mn @ ZStage @ np.transpose(Mm) @ np.transpose(Vm)
     px[len(u)*i:len(u)*(i+1),len(v)*j:len(v)*(j+1)]  = pxStage
     py[len(u)*i:len(u)*(i+1),len(v)*j:len(v)*(j+1)]  = pyStage
     pz[len(u)*i:len(u)*(i+1),len(v)*j:len(v)*(j+1)]  = pzStage
fig = go.Figure(go.Surface(x=X,y=Y,z=Z,opacity=.1,colorscale ='ylorrd'))
fig.add_trace(go.Surface(x=px,y=py,z=pz,opacity=1,colorscale='PuBu'))
fig.update_layout(scene = dict(aspectratio = dict(x=2, y=1.8, z=.5)))

**23.17. Solutions to exercises in Chapter 20**

In [ ]:
import numpy as np, plotly.graph_objects as go
u = np.linspace(0,2*np.pi,50)
r = 2 + np.sin(u*3)
X, Y, Z = r*np.cos(u).reshape(1,-1), r*np.sin(u).reshape(1,-1), 0*u.reshape(1,-1)
for i in range(2,6):
  r = 2 + np.sin(u*(i+2))
  x, y, z = r * np.cos(u), r * np.sin(u), u*0+i-1
  X = np.concatenate((X,x.reshape(1,-1)))
  Y = np.concatenate((Y,y.reshape(1,-1)))
  Z = np.concatenate((Z,z.reshape(1,-1)))
pxu0, pyu0, pzu0 = X[:,0].reshape(-1,1), Y[:,0].reshape(-1,1), Z[:,0].reshape(-1,1)
pxuend, pyuend, pzuend = X[:,-1].reshape(-1,1), Y[:,-1].reshape(-1,1), Z[:,-1].reshape(-1,1)
X, Y, Z = np.concatenate((pxu0,X,pxuend),axis=1), np.concatenate((pyu0,Y,pyuend),axis=1), np.concatenate((pzu0,Z,pzuend),axis=1)
pxv0, pyv0, pzv0 = X[0,:].reshape(1,-1), Y[0,:].reshape(1,-1), Z[0,:].reshape(1,-1)
pxvend, pyvend, pzvend = X[-1,:].reshape(1,-1), Y[-1,:].reshape(1,-1), Z[-1,:].reshape(1,-1)
X, Y, Z = np.concatenate((pxv0,X,pxvend)), np.concatenate((pyv0,Y,pyvend)), np.concatenate((pzv0,Z,pzvend))
u = np.transpose(np.linspace(0,1,4).reshape(1, -1))
v = np.transpose(np.linspace(0,1,4).reshape(1, -1))
Un = np.concatenate((v**3,v**2,v,np.ones(np.shape(v))),axis=1)
Vm = np.concatenate((v**3,v**2,v,np.ones(np.shape(v))),axis=1)
Mn = (1/2)*np.array([[-1, 3,-3, 1], [ 2,-5, 4,-1], [-1, 0, 1, 0], [ 0, 2, 0, 0]])
Mm = (1/2)*np.array([[-1, 3,-3, 1], [ 2,-5, 4,-1], [-1, 0, 1, 0], [ 0, 2, 0, 0]])
n, m = X.shape
px = np.zeros((len(u)*(n-len(Mn)+1), len(v)*(m-len(Mm)+1)))
py = np.zeros((len(u)*(n-len(Mn)+1), len(v)*(m-len(Mm)+1)))
pz = np.zeros((len(u)*(n-len(Mn)+1), len(v)*(m-len(Mm)+1)))
for i in range(n-len(Mn)+1):
   for j in range(m-len(Mm)+1):
      XStage = X[i:i+len(Mn),j:j+len(Mm)]
      YStage = Y[i:i+len(Mn),j:j+len(Mm)]
      ZStage = Z[i:i+len(Mn),j:j+len(Mm)]
      pxStage = Un @ Mn @ XStage @ np.transpose(Mm) @ np.transpose(Vm)
      pyStage = Un @ Mn @ YStage @ np.transpose(Mm) @ np.transpose(Vm)
      pzStage = Un @ Mn @ ZStage @ np.transpose(Mm) @ np.transpose(Vm)
      px[len(u)*i:len(u)*(i+1),len(v)*j:len(v)*(j+1)] = pxStage[:,:]
      py[len(u)*i:len(u)*(i+1),len(v)*j:len(v)*(j+1)] = pyStage[:,:]
      pz[len(u)*i:len(u)*(i+1),len(v)*j:len(v)*(j+1)] = pzStage[:,:]
fig = go.Figure(go.Surface(x=px,y=py,z=pz,opacity=1,colorscale='solar'))
fig.update_layout(scene = dict(aspectratio = dict(x=1, y=1, z=1)))